# LLM Judge Output Parser

## Purpose
Post-process LLM judge outputs to handle models that wrap JSON in markdown code fences.

## Problem
Some judge models (e.g., Google Gemini 2.5 Flash) return responses like:
```
```json
{"technical_accuracy": {"score": 20}, ...}
```
```

This causes JSON parsing to fail during the judging process, leaving `judge.fields` with null values.

## Solution
1. Strip markdown code fences from `judge.raw_output`
2. Re-parse the cleaned JSON
3. Populate `judge.fields` correctly
4. Write back to the same file (in-place update)

## Safety
- Preserves original `raw_output` (never modified)
- Atomic file writes (temp file + rename)
- Idempotent (safe to run multiple times)
- Git-trackable changes

# Configuration

In [ ]:
from pathlib import Path
import json
import re
from tqdm.auto import tqdm

# -----------------------------
# CONFIG
# -----------------------------
PHASE5_ROOT = Path(".") / "sysengbench-osq-llm-judge"

# Filter: Only process files matching this pattern
# Set to None to process ALL judge files
JUDGE_FILTER = "google_gemini-2.5-flash"  # Only Gemini files
# JUDGE_FILTER = None  # Uncomment to process all files

# Prompt format to handle
PROMPT_FORMAT = "p1"  # p1 = scores only (no justifications)

# Dry run mode (preview only, no writes)
DRY_RUN = True  # Set to False to actually write changes

print(f"Phase 5 Root: {PHASE5_ROOT}")
print(f"Judge Filter: {JUDGE_FILTER}")
print(f"Prompt Format: {PROMPT_FORMAT}")
print(f"Dry Run Mode: {DRY_RUN}")

# Helper Functions

In [ ]:
def clean_markdown_fences(raw_output: str) -> str:
    """
    Strip markdown code fences from LLM output.
    
    Handles patterns like:
    - ```json\n{...}\n```
    - ```\n{...}\n```
    
    Returns cleaned string (or original if no fences found).
    """
    if not raw_output:
        return raw_output
    
    cleaned = raw_output.strip()
    
    # Pattern: ```json\n{...}\n``` or ```\n{...}\n```
    fence_pattern = r'^```(?:json)?\s*\n?(.*?)\n?```\s*$'
    match = re.match(fence_pattern, cleaned, re.DOTALL)
    
    if match:
        return match.group(1).strip()
    
    return cleaned


def safe_json_parse(s: str) -> dict:
    """
    Parse JSON string, returning None on failure.
    """
    try:
        return json.loads(s)
    except:
        return None


def extract_fields_p1(parsed: dict) -> dict:
    """
    Extract fields for p1 format (scores only, no justifications).
    
    Expected parsed structure:
    {
      "technical_accuracy": {"score": 20},
      "conceptual_understanding": {"score": 19},
      ...
      "overall_score": 98
    }
    """
    def get_score(field_name):
        field = parsed.get(field_name)
        if isinstance(field, dict):
            return field.get("score")
        return None
    
    return {
        "technical_accuracy": {"score": get_score("technical_accuracy")},
        "conceptual_understanding": {"score": get_score("conceptual_understanding")},
        "completeness": {"score": get_score("completeness")},
        "clarity_organization": {"score": get_score("clarity_organization")},
        "professional_relevance": {"score": get_score("professional_relevance")},
        "overall_score": parsed.get("overall_score")
    }


def needs_cleaning(record: dict) -> bool:
    """
    Check if a record needs cleaning.
    
    Returns True if:
    - judge.fields.overall_score is None
    - judge.raw_output exists and contains markdown fences
    """
    judge = record.get("judge", {})
    fields = judge.get("fields", {})
    raw_output = judge.get("raw_output", "")
    
    # Already parsed correctly?
    if fields.get("overall_score") is not None:
        return False
    
    # Has markdown fences?
    if raw_output and "```json" in raw_output:
        return True
    
    return False


print("Helper functions loaded.")

# File Discovery

In [ ]:
def find_judge_files(root: Path, filter_pattern: str = None) -> list:
    """
    Find all judge files under root directory.
    
    Args:
        root: Root directory to search
        filter_pattern: Only include files matching this substring (e.g., "google_gemini")
    
    Returns:
        List of Path objects
    """
    files = []
    
    for model_dir in root.iterdir():
        if not model_dir.is_dir():
            continue
        
        for file in model_dir.glob("*.jsonl"):
            # Skip if doesn't match filter
            if filter_pattern and filter_pattern not in file.name:
                continue
            
            files.append(file)
    
    return sorted(files)


# Discover files
target_files = find_judge_files(PHASE5_ROOT, JUDGE_FILTER)

print(f"\nFound {len(target_files)} files to process:")
for f in target_files:
    print(f"  - {f.parent.name}/{f.name}")

# Preview Mode (Dry Run)

Check which records need cleaning without modifying files.

In [ ]:
def preview_file(filepath: Path) -> dict:
    """
    Preview which records need cleaning.
    
    Returns:
        {
            'total': int,
            'needs_cleaning': int,
            'already_ok': int,
            'no_raw_output': int,
            'sample_records': [list of sample_ids that need cleaning]
        }
    """
    stats = {
        'total': 0,
        'needs_cleaning': 0,
        'already_ok': 0,
        'no_raw_output': 0,
        'sample_records': []
    }
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            
            record = json.loads(line)
            stats['total'] += 1
            
            judge = record.get('judge', {})
            fields = judge.get('fields', {})
            raw_output = judge.get('raw_output')
            
            # Already parsed?
            if fields.get('overall_score') is not None:
                stats['already_ok'] += 1
                continue
            
            # No raw output?
            if not raw_output:
                stats['no_raw_output'] += 1
                continue
            
            # Needs cleaning?
            if needs_cleaning(record):
                stats['needs_cleaning'] += 1
                stats['sample_records'].append(record.get('sample_id'))
    
    return stats


# Preview all files
print("\n" + "="*60)
print("PREVIEW: Files needing cleaning")
print("="*60)

total_to_clean = 0

for filepath in target_files:
    stats = preview_file(filepath)
    total_to_clean += stats['needs_cleaning']
    
    print(f"\n{filepath.parent.name}/{filepath.name}")
    print(f"  Total records: {stats['total']}")
    print(f"  Already OK: {stats['already_ok']}")
    print(f"  Needs cleaning: {stats['needs_cleaning']}")
    print(f"  No raw output: {stats['no_raw_output']}")
    
    if stats['needs_cleaning'] > 0:
        print(f"  Sample IDs (first 10): {stats['sample_records'][:10]}")

print(f"\n{'='*60}")
print(f"TOTAL RECORDS TO CLEAN: {total_to_clean}")
print(f"{'='*60}")

# Execution: Clean and Update Files

**⚠️ WARNING**: This will modify files in-place.

Make sure:
1. You've reviewed the preview above
2. Git is clean or you've committed current state
3. `DRY_RUN = False` in config cell

Run this cell to perform the cleaning.

In [ ]:
def process_file(filepath: Path, dry_run: bool = True) -> dict:
    """
    Process a single judge file: clean markdown fences and update fields.
    
    Args:
        filepath: Path to judge file
        dry_run: If True, don't write changes (preview only)
    
    Returns:
        Stats dict with counts
    """
    records = []
    stats = {
        'total': 0,
        'cleaned': 0,
        'already_ok': 0,
        'failed': 0,
        'no_raw_output': 0
    }
    
    # Read all records
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            
            record = json.loads(line)
            stats['total'] += 1
            
            judge = record.get('judge', {})
            fields = judge.get('fields', {})
            raw_output = judge.get('raw_output')
            
            # Already parsed?
            if fields.get('overall_score') is not None:
                stats['already_ok'] += 1
                records.append(record)
                continue
            
            # No raw output?
            if not raw_output:
                stats['no_raw_output'] += 1
                records.append(record)
                continue
            
            # Try cleaning and parsing
            cleaned = clean_markdown_fences(raw_output)
            parsed = safe_json_parse(cleaned)
            
            if parsed:
                # Update fields (preserve raw_output)
                if PROMPT_FORMAT == "p1":
                    judge['fields'] = extract_fields_p1(parsed)
                else:
                    raise NotImplementedError(f"Prompt format {PROMPT_FORMAT} not implemented")
                
                stats['cleaned'] += 1
            else:
                stats['failed'] += 1
            
            records.append(record)
    
    # Write back (only if not dry run)
    if not dry_run:
        # Atomic write: temp file + rename
        temp_file = filepath.with_suffix('.tmp')
        with open(temp_file, 'w', encoding='utf-8') as f:
            for record in records:
                f.write(json.dumps(record) + '\n')
        
        # Replace original
        temp_file.replace(filepath)
    
    return stats


# Process all files
if DRY_RUN:
    print("\n⚠️  DRY RUN MODE - No files will be modified")
    print("Set DRY_RUN = False in config cell to apply changes\n")
else:
    print("\n✅ LIVE MODE - Files will be modified\n")

print("="*60)
print("Processing files...")
print("="*60)

total_stats = {
    'total': 0,
    'cleaned': 0,
    'already_ok': 0,
    'failed': 0,
    'no_raw_output': 0
}

for filepath in tqdm(target_files, desc="Processing files"):
    stats = process_file(filepath, dry_run=DRY_RUN)
    
    # Accumulate stats
    for key in total_stats:
        total_stats[key] += stats[key]
    
    # Print per-file results
    print(f"\n{filepath.parent.name}/{filepath.name}")
    print(f"  Cleaned: {stats['cleaned']} | Already OK: {stats['already_ok']} | Failed: {stats['failed']}")

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total records: {total_stats['total']}")
print(f"Cleaned: {total_stats['cleaned']}")
print(f"Already OK: {total_stats['already_ok']}")
print(f"Failed: {total_stats['failed']}")
print(f"No raw output: {total_stats['no_raw_output']}")
print(f"{'='*60}")

if DRY_RUN:
    print("\n⚠️  DRY RUN COMPLETE - No changes made")
else:
    print("\n✅ PROCESSING COMPLETE - Files updated")

# Validation: Verify Cleaning Success

Check that cleaned records now parse correctly.

In [ ]:
def validate_file(filepath: Path) -> dict:
    """
    Validate that all records in a file have proper field parsing.
    
    Returns:
        {
            'total': int,
            'valid': int (has overall_score),
            'invalid': int (missing overall_score),
            'invalid_ids': [list of sample_ids]
        }
    """
    stats = {
        'total': 0,
        'valid': 0,
        'invalid': 0,
        'invalid_ids': []
    }
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            
            record = json.loads(line)
            stats['total'] += 1
            
            fields = record.get('judge', {}).get('fields', {})
            
            if fields.get('overall_score') is not None:
                stats['valid'] += 1
            else:
                stats['invalid'] += 1
                stats['invalid_ids'].append(record.get('sample_id'))
    
    return stats


# Validate all files
print("\n" + "="*60)
print("VALIDATION: Check parsing success")
print("="*60)

total_valid = 0
total_invalid = 0

for filepath in target_files:
    stats = validate_file(filepath)
    total_valid += stats['valid']
    total_invalid += stats['invalid']
    
    print(f"\n{filepath.parent.name}/{filepath.name}")
    print(f"  Valid: {stats['valid']}/{stats['total']}")
    
    if stats['invalid'] > 0:
        print(f"  ⚠️  Invalid: {stats['invalid']}")
        print(f"  Invalid IDs (first 10): {stats['invalid_ids'][:10]}")

print(f"\n{'='*60}")
print(f"TOTAL VALID: {total_valid}")
print(f"TOTAL INVALID: {total_invalid}")
print(f"{'='*60}")

if total_invalid == 0:
    print("\n✅ All records validated successfully!")
else:
    print(f"\n⚠️  {total_invalid} records still need attention")

# Sample Inspection

Examine a few records to verify cleaning worked correctly.

In [ ]:
# Pick first file and show a sample cleaned record
if target_files:
    sample_file = target_files[0]
    
    print(f"Sample from: {sample_file.name}\n")
    
    with open(sample_file, 'r', encoding='utf-8') as f:
        # Find first cleaned record (has overall_score)
        for line in f:
            if not line.strip():
                continue
            
            record = json.loads(line)
            fields = record.get('judge', {}).get('fields', {})
            
            if fields.get('overall_score') is not None:
                print("Sample Record:")
                print(json.dumps({
                    'sample_id': record.get('sample_id'),
                    'judge': {
                        'fields': fields,
                        'raw_output': record.get('judge', {}).get('raw_output', '')[:200] + '...'
                    }
                }, indent=2))
                break
else:
    print("No files to sample.")